In [ ]:
# Facial Recognition Pipeline with Geotag Tracking
## Face Detection, Extraction & Similarity Search System

In [28]:
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from PIL.ExifTags import TAGS
import os
from pathlib import Path
import pickle
import json
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from scipy.spatial.distance import cosine
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [29]:
# Install required packages
import subprocess
import sys

# Package name to import name mapping
packages = [
    ('opencv-contrib-python', 'cv2'),
    ('deepface', 'deepface'),
    ('pillow', 'PIL'),
    ('pandas', 'pandas'),
    ('scipy', 'scipy')
]

for package, import_name in packages:
    try:
        __import__(import_name)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package} installed")

print("\nAll dependencies ready!")

✓ opencv-contrib-python already installed
✓ deepface already installed
✓ pillow already installed
✓ pandas already installed
✓ scipy already installed

All dependencies ready!


In [ ]:
from deepface import DeepFace

class FacialRecognitionPipeline:
    """
    Complete facial recognition pipeline with geotag tracking and facial search
    """

    def __init__(self, database_dir: str = "./face_database"):
        """
        Initialize the pipeline

        Args:
            database_dir: Directory to store face embeddings and metadata
        """
        self.database_dir = database_dir
        # Use DataFrame for structured storage: 512 embedding columns + metadata
        embedding_cols = [f'embedding_{i}' for i in range(512)]
        metadata_cols = ['latitude', 'longitude', 'altitude', 'timestamp', 'filename', 'person_id', 'face_location', 'confidence']
        self.faces_db = pd.DataFrame(columns=embedding_cols + metadata_cols)

        self.face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        )

        Path(database_dir).mkdir(exist_ok=True)
        self.db_file = os.path.join(database_dir, 'faces_database.pkl')
        self.metadata_file = os.path.join(database_dir, 'metadata.json')

        self.load_database()

    def extract_geotag(self, image_path: str) -> Dict:
        """
        Extract geotag (GPS coordinates) from image EXIF data

        Args:
            image_path: Path to the image file

        Returns:
            Dictionary with latitude, longitude, timestamp, and location info
        """
        geotag = {
            'latitude': None,
            'longitude': None,
            'altitude': None,
            'timestamp': None,
            'filename': os.path.basename(image_path)
        }

        try:
            image = Image.open(image_path)
            exif_data = image._getexif()

            if exif_data is not None:
                for tag_id, value in exif_data.items():
                    tag_name = TAGS.get(tag_id, tag_id)

                    # Extract GPS info
                    if tag_name == 'GPSInfo':
                        gps_data = {}
                        for t in value:
                            sub_tag_name = TAGS.get(t, t)
                            gps_data[sub_tag_name] = value[t]

                        if 'GPSLatitude' in gps_data and 'GPSLongitude' in gps_data:
                            geotag['latitude'] = self._convert_to_degrees(gps_data['GPSLatitude'])
                            geotag['longitude'] = self._convert_to_degrees(gps_data['GPSLongitude'])

                        if 'GPSAltitude' in gps_data:
                            geotag['altitude'] = float(gps_data['GPSAltitude'])

                    # Extract timestamp
                    elif tag_name == 'DateTime':
                        geotag['timestamp'] = str(value)

        except Exception as e:
            print(f"Warning: Could not extract EXIF data: {e}")

        # Add current timestamp if not found in EXIF
        if geotag['timestamp'] is None:
            geotag['timestamp'] = datetime.now().isoformat()

        return geotag

    @staticmethod
    def _convert_to_degrees(value):
        """Convert GPS coordinates to degrees"""
        d, m, s = value
        return d + (m / 60.0) + (s / 3600.0)

    def _extract_face_embeddings(self, image_path: str) -> List[Dict]:
        """
        Extract face embeddings from an image WITHOUT adding to database
        Used for search/query operations

        Args:
            image_path: Path to the image file

        Returns:
            List of dictionaries containing face embeddings and geotag
        """
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")

        # Extract geotag
        geotag = self.extract_geotag(image_path)

        # Load image
        image = cv2.imread(image_path)
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Detect faces
        faces = self.face_cascade.detectMultiScale(gray, 1.3, 5)
        print(f"DEBUG: extracted {len(faces)} face(s) from query image {image_path}")
        if len(faces) > 0:
            print(f"  rectangles: {faces}")

        face_data_list = []

        for idx, (x, y, w, h) in enumerate(faces):
            # Extract face region
            face_img = rgb_image[y:y+h, x:x+w]

            try:
                # Generate embedding using DeepFace
                embeddings = DeepFace.represent(face_img, model_name='Facenet512', enforce_detection=False)
                embedding = np.array(embeddings[0]['embedding'])

                face_record = {
                    'embedding': embedding,
                    'face_image': face_img,
                    'geotag': geotag,
                    'face_location': (x, y, w, h),
                    'confidence': embeddings[0].get('face_confidence', None)
                }

                face_data_list.append(face_record)

            except Exception as e:
                print(f"Warning: Could not process face {idx} in {image_path}: {e}")
                continue

        print(f"✓ Extracted {len(face_data_list)} face(s) from {image_path}")
        print(f"  Location: {geotag['latitude']}, {geotag['longitude']}")

        return face_data_list

    def detect_and_extract_faces(self, image_path: str, person_id: str = None) -> List[Dict]:
        """
        Detect faces in an image and extract their embeddings
        Adds faces to the database for registration

        Args:
            image_path: Path to the image file
            person_id: Identifier for the person (optional)

        Returns:
            List of dictionaries containing face data and geotag
        """
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")

        # Extract geotag
        geotag = self.extract_geotag(image_path)

        # Load image
        image = cv2.imread(image_path)
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Detect faces
        faces = self.face_cascade.detectMultiScale(gray, 1.3, 5)
        print(f"DEBUG: registration detected {len(faces)} face(s) in {image_path}")
        if len(faces) > 0:
            print(f"  rectangles: {faces}")

        face_data_list = []

        for idx, (x, y, w, h) in enumerate(faces):
            # Extract face region
            face_img = rgb_image[y:y+h, x:x+w]

            try:
                # Generate embedding using DeepFace
                embeddings = DeepFace.represent(face_img, model_name='Facenet512', enforce_detection=False)
                embedding = np.array(embeddings[0]['embedding'])

                # Create row data for DataFrame
                row_data = {}
                for i in range(512):
                    row_data[f'embedding_{i}'] = embedding[i]

                row_data.update({
                    'latitude': geotag['latitude'],
                    'longitude': geotag['longitude'],
                    'altitude': geotag['altitude'],
                    'timestamp': datetime.now().isoformat(),
                    'filename': geotag['filename'],
                    'person_id': person_id or f"person_{len(self.faces_db)}",
                    'face_location': (x, y, w, h),
                    'confidence': embeddings[0].get('face_confidence', None)
                })

                # Add to DataFrame
                self.faces_db = pd.concat([self.faces_db, pd.DataFrame([row_data])], ignore_index=True)

                face_record = {
                    'embedding': embedding,
                    'face_image': face_img,
                    'geotag': geotag,
                    'person_id': row_data['person_id'],
                    'face_location': row_data['face_location'],
                    'timestamp': row_data['timestamp'],
                    'confidence': row_data['confidence']
                }

                face_data_list.append(face_record)

            except Exception as e:
                print(f"Warning: Could not process face {idx} in {image_path}: {e}")
                continue

        print(f"✓ Found and processed {len(face_data_list)} face(s) in {image_path}")
        print(f"  Location: {geotag['latitude']}, {geotag['longitude']}")

        return face_data_list

    def search_similar_faces(self, query_image_path: str, similarity_threshold: float = 0.6) -> List[Dict]:
        """
        Search for similar faces in database

        Args:
            query_image_path: Path to query image
            similarity_threshold: Cosine similarity threshold (0-1, higher = more similar)

        Returns:
            List of matching faces with their geotags and similarity scores
        """
        if self.faces_db.empty:
            print("Database is empty! Please register faces first.")
            return []

        # Extract faces from query image (without adding to database)
        query_faces = self._extract_face_embeddings(query_image_path)

        if not query_faces:
            print("No faces detected in query image")
            return []

        results = []

        for query_face in query_faces:
            query_embedding = query_face['embedding']
            matches = []

            # Compare with all faces in database
            for idx, db_row in self.faces_db.iterrows():
                db_embedding = np.array([db_row[f'embedding_{i}'] for i in range(512)])

                # Calculate cosine similarity (1 - cosine distance)
                similarity = 1 - cosine(query_embedding, db_embedding)

                if similarity >= similarity_threshold:
                    matches.append({
                        'similarity_score': float(similarity),
                        'person_id': db_row['person_id'],
                        'geotag': {
                            'latitude': db_row['latitude'],
                            'longitude': db_row['longitude'],
                            'altitude': db_row['altitude'],
                            'timestamp': db_row['timestamp'],
                            'filename': db_row['filename']
                        },
                        'last_seen': db_row['timestamp'],
                        'face_location': db_row['face_location']
                    })

            # Sort by similarity score
            matches.sort(key=lambda x: x['similarity_score'], reverse=True)

            result = {
                'query_face': query_face,
                'matches': matches,
                'match_count': len(matches)
            }
            results.append(result)

        return results

    def save_database(self):
        """Save database to disk"""
        try:
            self.faces_db.to_pickle(self.db_file)

            # Save metadata separately
            metadata = {
                'total_faces': len(self.faces_db),
                'last_updated': datetime.now().isoformat(),
                'persons': self.faces_db['person_id'].unique().tolist()
            }
            with open(self.metadata_file, 'w') as f:
                json.dump(metadata, f, indent=2)

            print(f"✓ Database saved with {len(self.faces_db)} faces")
        except Exception as e:
            print(f"Error saving database: {e}")

    def load_database(self):
        """Load database from disk"""
        if os.path.exists(self.db_file):
            try:
                self.faces_db = pd.read_pickle(self.db_file)
                print(f"✓ Loaded {len(self.faces_db)} faces from database")
            except Exception as e:
                print(f"Error loading database: {e}")

    def get_person_geotags(self, person_id: str) -> pd.DataFrame:
        """
        Get all geotags for a specific person

        Args:
            person_id: ID of the person

        Returns:
            DataFrame with geotag history
        """
        person_faces = self.faces_db[self.faces_db['person_id'] == person_id]

        if person_faces.empty:
            return pd.DataFrame()

        geotag_history = person_faces[['latitude', 'longitude', 'timestamp', 'filename', 'altitude']].copy()
        return geotag_history

    def visualize_results(self, results: List[Dict], max_matches: int = 3):
        """Visualize search results"""
        for idx, result in enumerate(results):
            query_face = result['query_face']
            matches = result['matches'][:max_matches]

            print(f"\n{'='*60}")
            print(f"Query Face {idx+1}: Found {result['match_count']} match(es)")
            print(f"{'='*60}")
            
            # Show query image's geotag (where the photo was uploaded from)
            print(f"\n[Query Photo Location]")
            print(f"  File: {query_face['geotag']['filename']}")
            print(f"  Latitude: {query_face['geotag']['latitude']}")
            print(f"  Longitude: {query_face['geotag']['longitude']}")
            print(f"  Altitude: {query_face['geotag']['altitude']}")
            print(f"  Timestamp: {query_face['geotag']['timestamp']}")

            if not matches:
                print(f"\nNo matching faces found in database")
                continue

            print(f"\n{'-'*60}")
            print(f"Matching Faces in Database:")
            print(f"{'-'*60}")

            for rank, match in enumerate(matches, 1):
                print(f"\n[Match {rank}]")
                print(f"  Similarity: {match['similarity_score']:.4f} ({match['similarity_score']*100:.2f}%)")
                print(f"  Person ID: {match['person_id']}")
                print(f"  Last Seen Location:")
                print(f"    Latitude: {match['geotag']['latitude']}")
                print(f"    Longitude: {match['geotag']['longitude']}")
                print(f"    Altitude: {match['geotag']['altitude']}")
                print(f"    Timestamp: {match['geotag']['timestamp']}")

                print(f"    File: {match['geotag']['filename']}")
                print(f"    File: {match['geotag']['filename']}")
                print("✓ Pipeline class initialized successfully!")


print("✓ Pipeline class initialized successfully!")

✓ Pipeline class initialized successfully!


## Step 1: Initialize Pipeline

In [31]:
# Clear old database and initialize fresh pipeline
import shutil

# Remove old database if it exists
if os.path.exists("./face_database"):
    shutil.rmtree("./face_database")
    print("✓ Old database cleared")

# Initialize the pipeline fresh
pipeline = FacialRecognitionPipeline(database_dir="./face_database")
print("Pipeline ready!")

✓ Old database cleared
Pipeline ready!


## Step 2: Register Faces (User uploads a photo)
### Example: Register person's photo with geotag

**Note:** Replace `"path/to/image.jpg"` with your actual image path

In [32]:
# Example: Register faces from multiple users
# Replace with your actual image paths

'''image_paths = [
    # Add your image paths here
    # "path/to/user1.jpg",
    # "path/to/user2.jpg",
]

registered_faces = []

for img_path in image_paths:
    if os.path.exists(img_path):
        person_name = Path(img_path).stem  # Use filename as person ID
        faces = pipeline.detect_and_extract_faces(img_path, person_id=person_name)
        registered_faces.extend(faces)
        print(f"Registered: {person_name}\n")
    else:
        print(f"File not found: {img_path}")

# Save to database
pipeline.save_database()

print(f"\nTotal faces in database: {len(pipeline.faces_db)}")'''

# Example: Register faces from multiple users
# Replace with your actual image paths

image_paths = [
    r"C:\Users\ASUS\Pictures\New folder\photo_bts_face_recog.jpg",
    
    # Add more images as needed
]

registered_faces = []

for img_path in image_paths:
    if os.path.exists(img_path):
        person_name = Path(img_path).stem  # Use filename as person ID
        faces = pipeline.detect_and_extract_faces(img_path, person_id=person_name)
        registered_faces.extend(faces)
        print(f"Registered: {person_name}\n")
    else:
        print(f"File not found: {img_path}")

# Save to database
pipeline.save_database()

print(f"\nTotal faces in database: {len(pipeline.faces_db)}")

✓ Found and processed 8 face(s) in C:\Users\ASUS\Pictures\New folder\photo_bts_face_recog.jpg
  Location: None, None
Registered: photo_bts_face_recog

✓ Database saved with 8 faces

Total faces in database: 8


## Step 3: Search for Similar Faces
### Another user uploads a photo to find matches

**Input:** New photo from a user  
**Output:** Similar faces found + their last geotag location

In [38]:
# Search for similar faces with detailed debugging
# Replace with a query image path
query_image = r"C:\Users\ASUS\Pictures\New folder\OIP.jpg"

if os.path.exists(query_image):
    print(f"Database has {len(pipeline.faces_db)} registered faces")
    print(f"Database columns: {list(pipeline.faces_db.columns[:10])}...\n")
    
    # Search with adjustable similarity threshold
    # Lower threshold to 0.3-0.4 for same person/photo variations
    search_results = pipeline.search_similar_faces(query_image, similarity_threshold=0.4)
    
    # Display results
    pipeline.visualize_results(search_results, max_matches=8)
    
    # Debug: Show all similarity scores (not just matches above threshold)
    print("\n" + "="*60)
    print("DEBUG: All Similarity Scores")
    print("="*60)
    
    query_faces = pipeline._extract_face_embeddings(query_image)
    for q_idx, query_face in enumerate(query_faces):
        query_embedding = query_face['embedding']
        print(f"\nQuery Face {q_idx+1}:")
        
        scores = []
        for db_idx, db_row in pipeline.faces_db.iterrows():
            db_embedding = np.array([db_row[f'embedding_{i}'] for i in range(512)])
            similarity = 1 - cosine(query_embedding, db_embedding)
            scores.append((db_idx, similarity, db_row['person_id']))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        for rank, (db_idx, score, person_id) in enumerate(scores[:8], 1):
            print(f"  Rank {rank}: Similarity={score:.4f} ({score*100:.2f}%) | Person={person_id}")
else:
    print(f"Query image not found: {query_image}")

Database has 8 registered faces
Database columns: ['embedding_0', 'embedding_1', 'embedding_2', 'embedding_3', 'embedding_4', 'embedding_5', 'embedding_6', 'embedding_7', 'embedding_8', 'embedding_9']...

✓ Extracted 1 face(s) from C:\Users\ASUS\Pictures\New folder\OIP.jpg
  Location: None, None

Query Face 1: Found 3 match(es)

[Query Photo Location]
  File: OIP.jpg
  Latitude: None
  Longitude: None
  Altitude: None
  Timestamp: 2026-03-06T05:01:39.214643

------------------------------------------------------------
Matching Faces in Database:
------------------------------------------------------------

[Match 1]
  Similarity: 0.5446 (54.46%)
  Person ID: photo_bts_face_recog
  Last Seen Location:
    Latitude: None
    Longitude: None
    Altitude: None
    Timestamp: 2026-03-06T04:47:38.498294
    File: photo_bts_face_recog.jpg

[Match 2]
  Similarity: 0.5222 (52.22%)
  Person ID: photo_bts_face_recog
  Last Seen Location:
    Latitude: None
    Longitude: None
    Altitude: None


In [ ]:
# Diagnostic: Check Database & Embeddings
print("="*60)
print("DATABASE DIAGNOSTICS")
print("="*60)
print(f"\nTotal faces registered: {len(pipeline.faces_db)}")
print(f"Unique persons: {pipeline.faces_db['person_id'].unique()}")
print(f"\nDatabase shape: {pipeline.faces_db.shape}")
print(f"Database columns: {list(pipeline.faces_db.columns)}")

# Check for None values in geotags
print(f"\nGeotag Status:")
print(f"  Latitude - None count: {pipeline.faces_db['latitude'].isna().sum()}")
print(f"  Longitude - None count: {pipeline.faces_db['longitude'].isna().sum()}")

# Check embeddings
print(f"\nEmbedding Status:")
embedding_cols = [f'embedding_{i}' for i in range(512)]
for col in embedding_cols[:3]:  # Check first 3 embedding columns
    none_count = pipeline.faces_db[col].isna().sum()
    if none_count > 0:
        print(f"  {col} - None count: {none_count}")

print(f"\nFirst registered face details:")
if not pipeline.faces_db.empty:
    first_face = pipeline.faces_db.iloc[0]
    print(f"  Person ID: {first_face['person_id']}")
    print(f"  Filename: {first_face['filename']}")
    print(f"  Latitude: {first_face['latitude']}")
    print(f"  Longitude: {first_face['longitude']}")
    print(f"  Timestamp: {first_face['timestamp']}")
    print(f"  Embedding sample: {[first_face[f'embedding_{i}'] for i in range(5)]}")

## Step 4: Get Complete Geotag History for a Person

In [35]:
# Get geotag history for a specific person
person_id = "person_0"  # Change to your person ID

geotag_df = pipeline.get_person_geotags(person_id)

if not geotag_df.empty:
    print(f"Geotag history for {person_id}:")
    print(geotag_df)
    print(f"\nLast location:")
    last_location = geotag_df.iloc[-1]
    print(f"  Latitude: {last_location['latitude']}")
    print(f"  Longitude: {last_location['longitude']}")
    print(f"  Timestamp: {last_location['timestamp']}")
else:
    print(f"No geotags found for {person_id}")

No geotags found for person_0


## Advanced: Complete Example with Mock Data

In [36]:
def generate_sample_face_embeddings():
    """
    Generate sample face data for demonstration
    (when you have real images, use pipeline.detect_and_extract_faces() instead)
    """
    # Create a new pipeline for demo
    demo_pipeline = FacialRecognitionPipeline(database_dir="./face_database_demo")
    
    # Sample person 1 data
    person1_embedding = np.random.randn(512)  # FaceNet512 produces 512D embeddings
    person1_data = {
        'embedding': person1_embedding,
        'face_image': np.random.randint(0, 255, (160, 160, 3), dtype=np.uint8),
        'geotag': {
            'latitude': 40.7128,
            'longitude': -74.0060,
            'timestamp': '2026-03-01T10:00:00',
            'filename': 'person1_photo1.jpg'
        },
        'person_id': 'john_doe',
        'face_location': (10, 10, 150, 150),
        'timestamp': datetime.now().isoformat()
    }
    
    # Sample person 2 data
    person2_embedding = np.random.randn(512)
    person2_data = {
        'embedding': person2_embedding,
        'face_image': np.random.randint(0, 255, (160, 160, 3), dtype=np.uint8),
        'geotag': {
            'latitude': 51.5074,
            'longitude': -0.1278,
            'timestamp': '2026-03-02T14:30:00',
            'filename': 'person2_photo1.jpg'
        },
        'person_id': 'jane_smith',
        'face_location': (20, 20, 140, 140),
        'timestamp': datetime.now().isoformat()
    }
    
    demo_pipeline.faces_db = [person1_data, person2_data]
    demo_pipeline.save_database()
    
    print("✓ Demo database created with sample faces")
    return demo_pipeline

# Uncomment to generate demo data
# demo_pipeline = generate_sample_face_embeddings()